<a href="https://colab.research.google.com/github/OluwatifeAkinsola/RAG-Chatbot/blob/main/Building_a_Human_Like_Investment_Analysis_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


The Free Data Layer

---



In [ ]:
import requests
import json
from datetime import datetime

class SECFetcher:
    """Fetch SEC filings from EDGAR (100% free)."""

    BASE_URL = "https://data.sec.gov/api/xbrl/companyfacts/"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Investment Analyst Agent (your@email.com)"
        })

    def get_company_facts(self, ticker):
        """Get XBRL facts for a company."""
        # Get CIK from ticker (simplified)
        cik = self._get_cik(ticker)
        if not cik:
            return None

        url = f"{self.BASE_URL}CIK{cik}.json"
        response = self.session.get(url)
        if response.status_code == 200:
            return response.json()
        return None

    def get_financial_statements(self, ticker):
        """Extract key financial metrics."""
        facts = self.get_company_facts(ticker)
        if not facts:
            return None

        # Extract relevant metrics
        metrics = {
            "revenue": self._extract_metric(facts, "RevenueFromContractWithCustomerIncludingAssessedTax"),
            "net_income": self._extract_metric(facts, "NetIncomeLoss"),
            "total_assets": self._extract_metric(facts, "Assets"),
            "total_liabilities": self._extract_metric(facts, "Liabilities"),
            "cash_flow": self._extract_metric(facts, "NetCashProvidedByOperatingActivities")
        }
        return metrics

    def _extract_metric(self, facts, metric_name):
        """Extract a specific metric from XBRL facts."""
        if not facts or "facts" not in facts:
            return None
        units = facts.get("facts", {}).get("us-gaap", {})
        if metric_name in units:
            unit_data = units[metric_name]
            # Try USD first
            for unit_key in unit_data.get("units", {}):
                if unit_key.startswith("USD"):
                    values = unit_data["units"][unit_key]
                    if values:
                        return sorted(values, key=lambda x: x.get("end", ""))[-1]
        return None

    def _get_cik(self, ticker):
        """Convert ticker to CIK (simplified)."""
        # In production, you'd use a mapping service
        mapping = {
            "AAPL": "0000320193",
            "MSFT": "0000789019",
            "GOOGL": "0001652044",
            "DANGCEM": "0001234567",  # Placeholder
        }
        return mapping.get(ticker.upper())

1.2 Finviz (Free Fundamentals & Screening)


In [ ]:
import requests
from bs4 import BeautifulSoup
import re

class FinvizFetcher:
    """Fetch financial data from Finviz (free)."""

    BASE_URL = "https://finviz.com/quote.ashx?t="

    def get_fundamentals(self, ticker):
        """Scrape fundamental data from Finviz."""
        url = f"{self.BASE_URL}{ticker}"
        response = requests.get(url)

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # Parse fundamental data
        fundamentals = {}
        table = soup.find('table', {'class': 'snapshot-table2'})

        if table:
            rows = table.find_all('tr')
            for row in rows:
                cells = row.find_all('td')
                if len(cells) >= 2:
                    key = cells[0].text.strip()
                    value = cells[1].text.strip()
                    fundamentals[key] = value

        return fundamentals

    def get_screener_results(self, filters=None):
        """Get stocks based on screening filters."""
        # Free screener API
        base_url = "https://finviz.com/screener.ashx?v=111&f=cap_large"
        if filters:
            base_url += "&" + "&".join(filters)
        response = requests.get(base_url)
        # Parse results...
        return response.text

 Yahoo Finance (Price & Historical Data)


In [ ]:
import yfinance as yf
import pandas as pd

class YahooFinanceFetcher:
    """Fetch price and basic financial data from Yahoo Finance."""

    def get_stock_info(self, ticker):
        """Get comprehensive stock information."""
        stock = yf.Ticker(ticker)
        return {
            "info": stock.info,
            "financials": stock.financials,
            "balance_sheet": stock.balance_sheet,
            "cashflow": stock.cashflow,
            "history": stock.history(period="1y")
        }

    def get_key_metrics(self, ticker):
        """Extract key valuation metrics."""
        stock = yf.Ticker(ticker)
        info = stock.info

        return {
            "price": info.get('currentPrice', info.get('regularMarketPrice')),
            "pe_ratio": info.get('trailingPE'),
            "forward_pe": info.get('forwardPE'),
            "pb_ratio": info.get('priceToBook'),
            "ps_ratio": info.get('priceToSalesTrailing12Months'),
            "market_cap": info.get('marketCap'),
            "dividend_yield": info.get('dividendYield'),
            "target_mean_price": info.get('targetMeanPrice'),
            "recommendation": info.get('recommendationKey', 'Hold'),
            "revenue": info.get('totalRevenue'),
            "gross_margin": info.get('grossMargins'),
            "profit_margin": info.get('profitMargins'),
        }

FRED API (Macroeconomic Data)


In [ ]:
import requests

class FREDFetcher:
    """Fetch macroeconomic data from FRED."""

    BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

    def __init__(self, api_key):
        self.api_key = api_key

    def get_series_data(self, series_id, limit=100):
        """Get data for a specific FRED series."""
        params = {
            'series_id': series_id,
            'api_key': self.api_key,
            'file_type': 'json',
            'limit': limit
        }
        response = requests.get(self.BASE_URL, params=params)
        if response.status_code == 200:
            return response.json()
        return None

    def get_gdp(self):
        """Get GDP data (GDP)."""
        return self.get_series_data('GDP')

    def get_inflation(self):
        """Get CPI/inflation data (CPIAUCSL)."""
        return self.get_series_data('CPIAUCSL')

    def get_interest_rates(self):
        """Get fed funds rate (FEDFUNDS)."""
        return self.get_series_data('FEDFUNDS')

    def get_unemployment(self):
        """Get unemployment rate (UNRATE)."""
        return self.get_series_data('UNRATE')

 Intelligent RAG Retrieval
 1. Smart Chunking for Financial Documents


In [ ]:
# Install langchain and langchain-community if they're not already installed
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters # Ensure this package is installed

from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

class FinancialDocumentChunker:
    """
    Specialized chunker for financial documents.
    Prioritizes important sections like 'Risk Factors' and 'MD&A'.
    """

    def __init__(self, chunk_size=1500, overlap=200):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk_document(self, text):
        """Chunk a financial document intelligently."""
        # Split by section headings first
        sections = self._split_by_sections(text)

        chunks = []
        for section in sections:
            # For important sections, keep them intact
            if self._is_important_section(section['heading']):
                chunks.append(section)
            else:
                # For other sections, use recursive splitting
                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=self.chunk_size,
                    chunk_overlap=self.overlap,
                    separators=["\n\n", "\n", ". ", " ", ""]
                )
                for chunk in splitter.split_text(section['content']):
                    chunks.append({
                        'heading': section['heading'],
                        'content': chunk
                    })

        return chunks

    def _split_by_sections(self, text):
        """Split document by section headings."""
        # Look for common section markers in SEC filings
        section_markers = [
            r'(ITEM\s+\d+[A-Z]?\.\s+[A-Z\s]+)',
            r'(PART\s+[IVX]+\s+[A-Z\s]+)',
            r'(RISK\s+FACTORS)',
            r'(MANAGEMENT\'S\s+DISCUSSION\s+AND\s+ANALYSIS)',
            r'(BUSINESS)',
            r'(LEGAL\s+PROCEEDINGS)',
            r'(QUANTITATIVE\s+AND\s+QUALITATIVE\s+DISCLOSURES)',
            r'(FINANCIAL\s+STATEMENTS)'
        ]

        pattern = '|'.join(section_markers)
        matches = list(re.finditer(pattern, text, re.IGNORECASE))

        if not matches:
            return [{'heading': 'Document', 'content': text}]

        sections = []
        for i, match in enumerate(matches):
            start = match.start()
            end = matches[i+1].start() if i+1 < len(matches) else len(text)
            sections.append({
                'heading': match.group(1).strip(),
                'content': text[start:end].strip()
            })

        return sections

    def _is_important_section(self, heading):
        """Check if a section is important (keep intact)."""
        important = [
            'RISK FACTORS',
            'MANAGEMENT\'S DISCUSSION AND ANALYSIS',
            'FINANCIAL STATEMENTS',
            'QUANTITATIVE AND QUALITATIVE DISCLOSURES'
        ]
        return any(imp in heading.upper() for imp in important)

Vector Database Setup (Free)


In [ ]:
!pip install chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

class VectorStoreManager:
    """Manage vector stores for RAG (completely free)."""

    def __init__(self, persist_directory="./chroma_db"):
        self.persist_directory = persist_directory
        self.embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2"  # Free, local, runs on CPU
        )
        self.client = chromadb.PersistentClient(path=persist_directory)

    def create_collection(self, collection_name):
        """Create or get a collection."""
        return self.client.get_or_create_collection(name=collection_name)

    def add_documents(self, collection_name, documents, metadatas=None):
        """Add documents to vector store."""
        collection = self.create_collection(collection_name)

        # For ChromaDB direct API
        for i, doc in enumerate(documents):
            collection.add(
                documents=[doc['content']],
                metadatas=[{'source': doc.get('source', 'unknown'), 'heading': doc.get('heading', '')}],
                ids=[f"{collection_name}_{i}"]
            )

    def search(self, collection_name, query, k=5):
        """Search for relevant documents."""
        collection = self.create_collection(collection_name)
        results = collection.query(
            query_texts=[query],
            n_results=k
        )
        return results

Multi-Agent Orchestration
The Analyst Agent (Writes the Analysis)


In [ ]:
!pip install ollama
import ollama

class AnalystAgent:
    """The primary analyst agent that writes investment memos."""

    def __init__(self, model="llama3.2:3b"):
        self.model = model

    def analyze(self, company_data, financials, context):
        """Generate a comprehensive investment analysis."""

        prompt = f"""
You are a senior investment analyst at a top-tier financial institution. Your task is to write a comprehensive, data-driven investment memo.

## Company Overview
- Ticker: {company_data.get('ticker')}
- Sector: {company_data.get('sector', 'N/A')}
- Price: ${company_data.get('price', 'N/A')}
- Market Cap: ${company_data.get('market_cap', 'N/A')}

## Key Financial Metrics
- P/E Ratio: {financials.get('pe_ratio', 'N/A')}
- Price/Book: {financials.get('pb_ratio', 'N/A')}
- Revenue Growth: {financials.get('revenue_growth', 'N/A')}
- Profit Margin: {financials.get('profit_margin', 'N/A')}
- Debt/Equity: {financials.get('debt_to_equity', 'N/A')}

## Context
{context}

## Instructions
1. Write a balanced analysis covering:
   - Business overview and competitive position
   - Financial performance and key metrics
   - Growth drivers and risks
   - Valuation assessment
2. **IMPORTANT:** Every numerical claim MUST be supported by a citation to the source.
3. End with a clear recommendation (BUY/HOLD/SELL) and price target.
4. Maintain a professional, objective tone.

## Format
Write in clear sections with headings. Use bullet points for key points.
"""

        response = ollama.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

The Devil's Advocate Agent (Challenges the Analysis)


In [ ]:
class DevilAdvocateAgent:
    """Challenges the analyst's findings for balance."""

    def __init__(self, model="llama3.2:3b"):
        self.model = model

    def challenge(self, analysis):
        """Review and challenge the analysis."""

        prompt = f"""
You are a rigorous "devil's advocate" analyst. Your task is to review the following investment analysis and challenge its assumptions, identify weak points, and present counterarguments.

## Analysis to Review
{analysis}

## Instructions
1. Identify 3-5 key assumptions that might be wrong.
2. Flag any over-optimistic projections.
3. Highlight risks the analyst may have downplayed.
4. Present a balanced counter-perspective on the recommendation.
5. Be constructive—your goal is to make the analysis stronger, not to destroy it.
6. Cite specific areas where you disagree and explain why.

## Format
- Section 1: Key Assumptions to Challenge
- Section 2: Overlooked Risks
- Section 3: Counter-Perspective on Recommendation
"""

        response = ollama.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

The Citation Validator Agent


In [ ]:
class CitationValidatorAgent:
    """Validates that all numerical claims are cited."""

    def __init__(self, model="llama3.2:3b"):
        self.model = model

    def validate(self, analysis, sources):
        """Check that all numbers are cited from sources."""

        prompt = f"""
You are a citation validator. Your task is to check that every numerical claim in the analysis has a proper source citation.

## Analysis
{analysis}

## Available Sources
{json.dumps(sources, indent=2)}

## Instructions
1. Extract all numerical claims from the analysis.
2. For each claim, check if it has a citation.
3. Flag any claim without a citation with [CITATION NEEDED].
4. Ensure citations match the sources provided.
5. Return the analysis with citation flags added.

## IMPORTANT RULES
- Any number (percentage, dollar amount, ratio, etc.) MUST have a citation.
- If a claim cannot be verified, mark it as [UNVERIFIED].
- If the citation is incorrect, mark it as [CITATION ERROR].
"""

        response = ollama.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']

 The Orchestrator


In [ ]:
class InvestmentAnalysisOrchestrator:
    """Orchestrates the entire analysis pipeline."""

    def __init__(self):
        self.analyst = AnalystAgent()
        self.devils_advocate = DevilAdvocateAgent()
        self.citation_validator = CitationValidatorAgent()
        self.sec_fetcher = SECFetcher()
        self.yahoo_fetcher = YahooFinanceFetcher()
        self.finviz_fetcher = FinvizFetcher()
        self.vector_store = VectorStoreManager()

    def analyze(self, ticker, query=None):
        """Run the complete investment analysis pipeline."""

        print(f"🔍 Starting analysis for {ticker}...")

        # Step 1: Gather data
        print("📊 Gathering data from multiple sources...")
        company_data = self.yahoo_fetcher.get_key_metrics(ticker)
        financials = self.sec_fetcher.get_financial_statements(ticker)
        fundamentals = self.finviz_fetcher.get_fundamentals(ticker)

        # Step 2: Retrieve relevant context (RAG)
        print("📚 Retrieving relevant documents...")
        context = self._get_relevant_context(ticker, query)

        # Step 3: Generate initial analysis
        print("✍️ Generating initial analysis...")
        analysis = self.analyst.analyze(company_data, financials, context)

        # Step 4: Devil's advocate review
        print("👿 Running devil's advocate review...")
        challenge = self.devils_advocate.challenge(analysis)

        # Step 5: Validate citations
        print("🔍 Validating citations...")
        validated_analysis = self.citation_validator.validate(analysis, {'sources': []})

        # Step 6: Synthesize final output
        print("📝 Synthesizing final report...")
        final_output = self._synthesize_report(
            analysis, challenge, validated_analysis, company_data
        )

        return final_output

    def _get_relevant_context(self, ticker, query):
        """Retrieve relevant context from vector store."""
        # In production, you would:
        # 1. Search vector DB for relevant documents
        # 2. Format them as context
        return "Context retrieved from filings and news sources..."

    def _synthesize_report(self, analysis, challenge, validated, data):
        """Combine all pieces into final report."""

        return f"""
# Investment Analysis Report: {data.get('ticker', 'N/A')}

## Executive Summary
[Overview of the company and key findings]

## Financial Analysis
{analysis}

## Risk Assessment (Devil's Advocate Review)
{challenge}

## Verified Analysis
{validated}

## Recommendation
{self._generate_recommendation(data)}

---
*Generated by Investment Analysis Agent | Data sourced from SEC EDGAR, Finviz, and Yahoo Finance*
"""

    def _generate_recommendation(self, data):
        """Generate a final recommendation based on data."""
        # Simple logic based on key metrics
        pe = data.get('pe_ratio', 0)
        growth = data.get('revenue_growth', 0)

        if pe < 15 and growth > 10:
            return "✅ BUY: Stock appears undervalued with strong growth."
        elif pe > 30 and growth < 5:
            return "❌ SELL: Stock appears overvalued with weak growth."
        else:
            return "➡️ HOLD: Stock is fairly valued. Wait for clearer signals."

In [ ]:
#!/usr/bin/env python3
"""
Investment Analysis Agent - Complete Production-Ready Version
Built with RAG, Multi-Agent Orchestration, and Free APIs
"""

import os
import sys
import json
import re
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
import requests
import yfinance as yf
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIGURATION
# ============================================================

@dataclass
class Config:
    """Central configuration for the agent."""

    # Model settings
    embedding_model: str = "all-MiniLM-L6-v2"
    llm_model: str = "llama-3.1-70b-versatile"  # Groq model
    temperature: float = 0.3
    max_tokens: int = 2048

    # RAG settings
    chunk_size: int = 1500
    chunk_overlap: int = 200
    top_k_retrieval: int = 5

    # API keys (set via environment variables)
    groq_api_key: Optional[str] = None

    # Paths
    persist_directory: str = "./chroma_db"

    def __post_init__(self):
        """Load API keys from environment."""
        self.groq_api_key = os.environ.get("GROQ_API_KEY")
        if not self.groq_api_key:
            print("⚠️ Warning: GROQ_API_KEY not set in environment")
            print("   Set it with: os.environ['GROQ_API_KEY'] = 'your-key'")

config = Config()

# ============================================================
# DEPENDENCY CHECKER
# ============================================================

def check_dependencies():
    """Check and install required dependencies."""
    required = [
        "yfinance", "requests", "beautifulsoup4", "pandas",
        "chromadb", "sentence-transformers", "langchain",
        "langchain-community", "langchain-text-splitters"
    ]

    missing = []
    for pkg in required:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"Installing missing dependencies: {', '.join(missing)}")
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        print("✅ Dependencies installed!")
    else:
        print("✅ All dependencies available.")

# ============================================================
# DATA FETCHERS
# ============================================================

class DataFetcher:
    """Unified data fetcher with fallbacks."""

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Investment Analysis Agent (contact@example.com)"
        })

    def get_company_data(self, ticker: str) -> Dict[str, Any]:
        """Fetch comprehensive company data from multiple sources."""

        result = {
            "ticker": ticker.upper(),
            "price": 0,
            "pe_ratio": 0,
            "pb_ratio": 0,
            "ps_ratio": 0,
            "market_cap": 0,
            "dividend_yield": 0,
            "profit_margin": 0,
            "revenue": 0,
            "revenue_growth": 0,
            "eps": 0,
            "target_price": 0,
            "recommendation": "Hold",
            "sector": "N/A",
            "industry": "N/A",
            "error": None
        }

        try:
            stock = yf.Ticker(ticker.upper())
            info = stock.info

            # Extract data with fallbacks
            result["price"] = info.get('currentPrice') or info.get('regularMarketPrice', 0)
            result["pe_ratio"] = info.get('trailingPE', 0) or 0
            result["pb_ratio"] = info.get('priceToBook', 0) or 0
            result["ps_ratio"] = info.get('priceToSalesTrailing12Months', 0) or 0
            result["market_cap"] = info.get('marketCap', 0) or 0
            result["dividend_yield"] = (info.get('dividendYield', 0) or 0) * 100
            result["profit_margin"] = (info.get('profitMargins', 0) or 0) * 100
            result["revenue"] = info.get('totalRevenue', 0) or 0
            result["eps"] = info.get('trailingEps', 0) or 0
            result["target_price"] = info.get('targetMeanPrice', 0) or 0
            result["recommendation"] = info.get('recommendationKey', 'Hold') or 'Hold'
            result["sector"] = info.get('sector', 'N/A') or 'N/A'
            result["industry"] = info.get('industry', 'N/A') or 'N/A'

            # Calculate revenue growth from financials
            try:
                financials = stock.financials
                if not financials.empty and 'Total Revenue' in financials.index:
                    revenue_data = financials.loc['Total Revenue'].tail(4)
                    if len(revenue_data) >= 2:
                        result["revenue_growth"] = (
                            (revenue_data.iloc[-1] - revenue_data.iloc[-2]) /
                            revenue_data.iloc[-2] * 100
                        )
            except:
                pass

        except Exception as e:
            result["error"] = str(e)

        return result

# ============================================================
# RAG SYSTEM
# ============================================================

class RAGSystem:
    """Retrieval-Augmented Generation system for financial documents."""

    def __init__(self, config: Config):
        self.config = config
        self.embeddings = None
        self.vector_store = None
        self._initialize()

    def _initialize(self):
        """Initialize embeddings and vector store."""
        try:
            from sentence_transformers import SentenceTransformer
            self.embeddings = SentenceTransformer(self.config.embedding_model)
        except ImportError:
            print("⚠️ sentence-transformers not available. RAG will be limited.")
            self.embeddings = None

    def add_documents(self, documents: List[str], metadatas: List[Dict] = None):
        """Add documents to the vector store."""
        if not self.embeddings:
            return

        try:
            import chromadb
            from chromadb.utils import embedding_functions

            self.client = chromadb.PersistentClient(path=self.config.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name="financial_docs",
                embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                    model_name=self.config.embedding_model
                )
            )

            # Add documents in batches
            if metadatas is None:
                metadatas = [{"source": "unknown"} for _ in documents]

            ids = [f"doc_{i}" for i in range(len(documents))]
            self.collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            print(f"✅ Added {len(documents)} documents to vector store")

        except Exception as e:
            print(f"⚠️ Vector store error: {e}")

    def retrieve(self, query: str, k: int = None) -> List[Dict]:
        """Retrieve relevant documents for a query."""
        if not self.embeddings or not hasattr(self, 'collection'):
            return []

        try:
            results = self.collection.query(
                query_texts=[query],
                n_results=k or self.config.top_k_retrieval
            )

            documents = []
            if results and results.get('documents'):
                for i, doc in enumerate(results['documents'][0]):
                    metadata = results.get('metadatas', [{}])[0][i] if results.get('metadatas') else {}
                    documents.append({
                        'content': doc,
                        'metadata': metadata,
                        'distance': results.get('distances', [[0]])[0][i] if results.get('distances') else 0
                    })

            return documents

        except Exception as e:
            print(f"⚠️ Retrieval error: {e}")
            return []

# ============================================================
# LLM CLIENT (Groq)
# ============================================================

class LLMClient:
    """Client for interacting with LLMs via Groq API."""

    def __init__(self, config: Config):
        self.config = config
        self.client = None
        self._initialize()

    def _initialize(self):
        """Initialize the Groq client."""
        try:
            from groq import Groq
            if self.config.groq_api_key:
                self.client = Groq(api_key=self.config.groq_api_key)
                print("✅ Groq client initialized")
            else:
                print("⚠️ No Groq API key found. LLM features disabled.")
        except ImportError:
            print("⚠️ Groq library not installed. Install with: pip install groq")

    def generate(self, prompt: str, temperature: float = None) -> str:
        """Generate a response from the LLM."""
        if not self.client:
            return "Error: LLM client not initialized."

        try:
            response = self.client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=self.config.llm_model,
                temperature=temperature or self.config.temperature,
                max_tokens=self.config.max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_with_context(self, system_prompt: str, user_prompt: str) -> str:
        """Generate a response with system context."""
        if not self.client:
            return "Error: LLM client not initialized."

        try:
            response = self.client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                model=self.config.llm_model,
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

# ============================================================
# ANALYST AGENTS
# ============================================================

class AnalystAgent:
    """Primary analyst agent for investment analysis."""

    def __init__(self, llm_client: LLMClient):
        self.llm = llm_client

    def analyze(self, data: Dict, context: List[Dict] = None) -> str:
        """Generate a comprehensive investment analysis."""

        context_text = ""
        if context:
            context_text = "\n\n".join([doc['content'] for doc in context[:5]])

        prompt = f"""
You are a senior investment analyst at a top-tier financial institution. Write a comprehensive investment memo.

## Company Overview
- Ticker: {data.get('ticker', 'N/A')}
- Sector: {data.get('sector', 'N/A')}
- Industry: {data.get('industry', 'N/A')}
- Price: ${data.get('price', 0):.2f}
- Market Cap: ${data.get('market_cap', 0):,.0f}

## Key Financial Metrics
- P/E Ratio: {data.get('pe_ratio', 0):.2f}
- P/B Ratio: {data.get('pb_ratio', 0):.2f}
- P/S Ratio: {data.get('ps_ratio', 0):.2f}
- Profit Margin: {data.get('profit_margin', 0):.1f}%
- Dividend Yield: {data.get('dividend_yield', 0):.2f}%
- Revenue Growth: {data.get('revenue_growth', 0):.1f}%
- EPS: ${data.get('eps', 0):.2f}
- Revenue: ${data.get('revenue', 0):,.0f}

## Analyst Consensus
- Recommendation: {data.get('recommendation', 'Hold')}
- Target Price: ${data.get('target_price', 0):.2f}

## Retrieved Context
{context_text}

## Instructions
1. Write a balanced, data-driven analysis covering:
   - Business overview and competitive position
   - Financial performance analysis
   - Growth drivers and key risks
   - Valuation assessment
2. **Every numerical claim must be supported by a citation**
3. End with a clear BUY/HOLD/SELL recommendation

## Format
Use clear sections with headings. Be concise but comprehensive.
"""

        system_prompt = """You are a senior investment analyst with 20+ years of experience.
You write clear, objective, and data-driven analysis. You never invent numbers or facts.
You always cite sources for your claims."""

        return self.llm.generate_with_context(system_prompt, prompt)

# ============================================================
# DEVIL'S ADVOCATE AGENT
# ============================================================

class DevilAdvocateAgent:
    """Agent that challenges the analysis."""

    def __init__(self, llm_client: LLMClient):
        self.llm = llm_client

    def challenge(self, analysis: str) -> str:
        """Review and challenge the analysis."""

        prompt = f"""
Review this investment analysis and challenge its assumptions:

## Analysis
{analysis}

## Instructions
1. Identify 3-5 key assumptions that could be wrong
2. Flag any over-optimistic projections
3. Highlight overlooked risks
4. Present a balanced counter-perspective
5. Be constructive - strengthen the analysis

## Format
- Section 1: Key Assumptions to Challenge
- Section 2: Overlooked Risks
- Section 3: Counter-Perspective on Recommendation
"""

        system_prompt = """You are a rigorous, skeptical analyst who plays "devil's advocate."
Your goal is to challenge assumptions and identify weaknesses. You are constructive, not destructive."""

        return self.llm.generate_with_context(system_prompt, prompt)

# ============================================================
# EXECUTIVE SUMMARY AGENT
# ============================================================

class ExecutiveSummaryAgent:
    """Agent that synthesizes the final report."""

    def __init__(self, llm_client: LLMClient):
        self.llm = llm_client

    def synthesize(self, analysis: str, challenge: str, data: Dict) -> str:
        """Synthesize the final executive summary."""

        prompt = f"""
Synthesize this investment analysis into a clear, actionable executive summary:

## Analysis
{analysis}

## Devil's Advocate Review
{challenge}

## Company Data
- Ticker: {data.get('ticker', 'N/A')}
- Price: ${data.get('price', 0):.2f}
- Market Cap: ${data.get('market_cap', 0):,.0f}

## Instructions
Create a 3-paragraph executive summary covering:
1. Investment thesis (2-3 sentences)
2. Key metrics and valuation
3. Final recommendation and conviction level
"""

        system_prompt = """You are a senior analyst synthesizing reports for busy executives.
Be concise, clear, and action-oriented. Every recommendation must be based on data."""

        return self.llm.generate_with_context(system_prompt, prompt)

# ============================================================
# ORCHESTRATOR
# ============================================================

class InvestmentOrchestrator:
    """Orchestrates the entire analysis pipeline."""

    def __init__(self, config: Config):
        self.config = config
        self.data_fetcher = DataFetcher()
        self.rag = RAGSystem(config)
        self.llm = LLMClient(config)
        self.analyst = AnalystAgent(self.llm)
        self.devil = DevilAdvocateAgent(self.llm)
        self.executive = ExecutiveSummaryAgent(self.llm)

    def analyze(self, ticker: str, query: str = None) -> Dict[str, Any]:
        """Run the complete investment analysis pipeline."""

        result = {
            "ticker": ticker.upper(),
            "timestamp": datetime.now().isoformat(),
            "status": "running"
        }

        try:
            print(f"\n🔍 Analyzing {ticker.upper()}...")

            # Step 1: Fetch data
            print("📊 Fetching company data...")
            data = self.data_fetcher.get_company_data(ticker)
            if data.get("error"):
                result["status"] = "error"
                result["error"] = data["error"]
                return result

            # Step 2: Retrieve context (RAG)
            print("📚 Retrieving relevant context...")
            context = []
            if query:
                context = self.rag.retrieve(query)

            # Step 3: Generate analysis
            print("✍️ Generating analysis...")
            analysis = self.analyst.analyze(data, context)

            # Step 4: Devil's advocate review
            print("👿 Running devil's advocate...")
            challenge = self.devil.challenge(analysis)

            # Step 5: Executive summary
            print("📝 Creating executive summary...")
            summary = self.executive.synthesize(analysis, challenge, data)

            # Step 6: Generate recommendation
            recommendation = self._generate_recommendation(data)

            result.update({
                "status": "success",
                "company_data": data,
                "analysis": analysis,
                "devils_advocate": challenge,
                "executive_summary": summary,
                "recommendation": recommendation,
                "context_used": len(context)
            })

            print("✅ Analysis complete!\n")

        except Exception as e:
            result["status"] = "error"
            result["error"] = str(e)

        return result

    def _generate_recommendation(self, data: Dict) -> Dict:
        """Generate a recommendation based on metrics."""
        pe = data.get('pe_ratio', 0)
        growth = data.get('revenue_growth', 0)
        profit = data.get('profit_margin', 0)

        if pe <= 0 or growth <= 0:
            return {
                "action": "HOLD",
                "confidence": "Low",
                "reason": "Insufficient data for confident recommendation"
            }

        score = 0
        if pe < 15:
            score += 2  # Undervalued
        elif pe < 25:
            score += 1  # Fairly valued
        else:
            score -= 1  # Overvalued

        if growth > 15:
            score += 2  # Strong growth
        elif growth > 5:
            score += 1  # Moderate growth

        if profit > 15:
            score += 1  # High profit margin

        if score >= 3:
            action = "BUY"
            reason = "Strong undervaluation and growth potential"
        elif score >= 0:
            action = "HOLD"
            reason = "Fairly valued with moderate prospects"
        else:
            action = "SELL"
            reason = "Overvalued with limited growth prospects"

        confidence = "High" if abs(score) >= 3 else "Medium" if abs(score) >= 1 else "Low"

        return {
            "action": action,
            "confidence": confidence,
            "reason": reason,
            "score": score
        }

# ============================================================
# DISPLAY & REPORTING
# ============================================================

def display_report(result: Dict):
    """Display the analysis report in a readable format."""

    if result.get("status") == "error":
        print(f"\n❌ Error: {result.get('error', 'Unknown error')}")
        return

    data = result.get("company_data", {})
    rec = result.get("recommendation", {})

    print("\n" + "=" * 70)
    print(f"📈 INVESTMENT ANALYSIS: {result.get('ticker', 'N/A')}")
    print("=" * 70)

    # Executive Summary
    print("\n📋 EXECUTIVE SUMMARY")
    print("-" * 50)
    print(result.get("executive_summary", "No summary available"))

    # Key Metrics
    print("\n📊 KEY METRICS")
    print("-" * 50)
    metrics = [
        ("Price", f"${data.get('price', 0):.2f}"),
        ("Market Cap", f"${data.get('market_cap', 0):,.0f}"),
        ("P/E Ratio", f"{data.get('pe_ratio', 0):.2f}"),
        ("P/B Ratio", f"{data.get('pb_ratio', 0):.2f}"),
        ("Profit Margin", f"{data.get('profit_margin', 0):.1f}%"),
        ("Revenue Growth", f"{data.get('revenue_growth', 0):.1f}%"),
        ("Dividend Yield", f"{data.get('dividend_yield', 0):.2f}%"),
        ("Analyst Consensus", data.get('recommendation', 'Hold')),
        ("Target Price", f"${data.get('target_price', 0):.2f}")
    ]
    for label, value in metrics:
        print(f"  {label:<20}: {value}")

    # Recommendation
    print("\n🎯 RECOMMENDATION")
    print("-" * 50)
    action = rec.get("action", "HOLD")
    if action == "BUY":
        emoji = "✅"
        color = "GREEN"
    elif action == "SELL":
        emoji = "❌"
        color = "RED"
    else:
        emoji = "➡️"
        color = "YELLOW"

    print(f"  {emoji} Action: {action}")
    print(f"  📊 Confidence: {rec.get('confidence', 'Medium')}")
    print(f"  💡 Reason: {rec.get('reason', 'No specific reason')}")

    # Analysis
    print("\n🔍 DETAILED ANALYSIS")
    print("-" * 50)
    print(result.get("analysis", "No analysis available"))

    # Devil's Advocate
    print("\n👿 DEVIL'S ADVOCATE REVIEW")
    print("-" * 50)
    print(result.get("devils_advocate", "No challenge available"))

    # Footer
    print("\n" + "=" * 70)
    print(f"Generated by Investment Analysis Agent")
    print(f"Timestamp: {result.get('timestamp', 'N/A')}")
    print(f"Context Retrieved: {result.get('context_used', 0)} documents")
    print("=" * 70)

# ============================================================
# MAIN
# ============================================================

def main():
    """Main entry point for the investment analysis agent."""

    print("=" * 70)
    print("🤖 INVESTMENT ANALYSIS AGENT")
    print("   Production-Ready | RAG + Multi-Agent | 100% Free")
    print("=" * 70)

    # Check dependencies
    print("\n📦 Checking dependencies...")
    check_dependencies()

    # Check API key
    if not config.groq_api_key:
        print("\n⚠️ GROQ_API_KEY not found.")
        print("   Please set it in your environment:")
        print("   os.environ['GROQ_API_KEY'] = 'your-api-key'")
        print("\n   Get a free key at: console.groq.com")
        return

    # Initialize orchestrator
    orchestrator = InvestmentOrchestrator(config)

    # Get user input
    print("\n" + "=" * 70)
    ticker = input("📈 Enter stock ticker (e.g., AAPL, MSFT, DANGCEM): ").strip().upper()
    if not ticker:
        print("❌ No ticker provided. Exiting.")
        return

    query = input("💬 Ask a question (optional, press Enter to skip): ").strip()

    # Run analysis
    result = orchestrator.analyze(ticker, query)

    # Display results
    display_report(result)

    # Save to file
    if result.get("status") == "success":
        filename = f"analysis_{ticker.lower()}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        with open(filename, 'w') as f:
            f.write(f"Ticker: {ticker}\n")
            f.write(f"Timestamp: {datetime.now().isoformat()}\n")
            f.write("=" * 70 + "\n\n")
            f.write(result.get("executive_summary", ""))
            f.write("\n\n" + "=" * 70 + "\n")
            f.write(result.get("analysis", ""))
            f.write("\n\n" + "=" * 70 + "\n")
            f.write(result.get("devils_advocate", ""))
        print(f"\n📄 Report saved to: {filename}")

if __name__ == "__main__":
    main()

⚠️ Warning: GROQ_API_KEY not set in environment
   Set it with: os.environ['GROQ_API_KEY'] = 'your-key'
🤖 INVESTMENT ANALYSIS AGENT
   Production-Ready | RAG + Multi-Agent | 100% Free

📦 Checking dependencies...
Installing missing dependencies: beautifulsoup4
✅ Dependencies installed!

⚠️ GROQ_API_KEY not found.
   Please set it in your environment:
   os.environ['GROQ_API_KEY'] = 'your-api-key'

   Get a free key at: console.groq.com
